# 04 -- Session and news analysis

Paired script: `analysis/join_news_events.py`. Independently recomputes `NEWS_BLACKOUT`
status (per `NewsManager.mqh`/section 10) for every journal decision, directly useful given
every real journal record's `news_state` is currently always empty (the live EA never sets
it -- see `analysis/schema.py`'s docstring).

**Uses clearly-labelled SYNTHETIC journal + news-event fixtures.** Real-data run: PENDING.

In [ ]:
import json
import sys
import tempfile
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from analysis.join_news_events import run

In [ ]:
tmp_dir = Path(tempfile.mkdtemp(prefix="themba_news_demo_"))

decision = {
    "signal_id": "sig-1", "timestamp_utc": "2026-07-21T14:05:30Z", "symbol": "XAUUSD",
    "market_family": "METAL", "intraday_mode": "SCALP", "regime": "REGIME_TRENDING_UP",
    "regime_confidence": 72.5, "direction": "BUY", "strategy": "TrendFollowingStrategy",
    "setup": "TrendlinePullback", "candlestick_pattern": None, "chart_pattern": None,
    "score": 68.0, "score_breakdown": {}, "entry": 2350.55, "stop": 2345.10,
    "targets": [2361.45], "risk_percent": 0.3, "news_state": "", "session_state": "",
    "reasons_passed": [], "reasons_rejected": [], "ea_version": "1.01", "git_commit": "abc",
}
(tmp_dir / "decisions_20260721.jsonl").write_text(json.dumps(decision) + "\n", encoding="utf-8")

pd.DataFrame([{
    "event_id": "e-nfp", "event_name": "NFP", "currency": "USD", "importance": 2,
    "scheduled_utc": "2026-07-21T14:10:00Z",  # 4m30s after the decision -- inside the window
}]).to_csv(tmp_dir / "news.csv", index=False)

In [ ]:
result = run(tmp_dir, tmp_dir / "news.csv", currency="USD", before_minutes=15, after_minutes=15,
             min_importance=2, output_csv=tmp_dir / "joined.csv", repo_path=PROJECT_ROOT.parents[1])

print(f"n_decisions      = {result.n_decisions}")
print(f"n_in_blackout    = {result.n_in_blackout}")
print(result.joined[["symbol", "timestamp_utc", "in_news_blackout", "triggering_event_id"]])

assert result.n_in_blackout == 1
assert result.joined.iloc[0]["triggering_event_id"] == "e-nfp"

## Real-data run: PENDING

Requires a real journal (batched runtime verification, TASK-025+) and a real news-event export -- neither exists yet.